# 01. Setup & Baseline

**Phase 1-2**: 환경 세팅 → DQ4AI 클론 → 데이터 다운로드 → DSC 엔진 정의 → 베이스라인 DSC & 모델 성능

---

## 0. 환경 설정

In [1]:
# ============================================================
# 0-1. Google Drive 마운트
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# 기본 경로 설정
import os

BASE = '/content/drive/MyDrive/capstone/dsc'
RAW_DIR = f'{BASE}/data/raw'
POLLUTED_DIR = f'{BASE}/data/polluted'
RESULTS_DIR = f'{BASE}/results'
DQ4AI_DIR = f'{BASE}/dq4ai'

for d in [RAW_DIR, POLLUTED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'BASE: {BASE}')
print('디렉토리 준비 완료')

Mounted at /content/drive
BASE: /content/drive/MyDrive/capstone/dsc
디렉토리 준비 완료


In [2]:
# ============================================================
# 0-2. 의존성 설치 & DQ4AI 클론
# ============================================================
%pip install -q xgboost

# DQ4AI 클론 (이미 있으면 스킵)
if not os.path.exists(DQ4AI_DIR):
    !git clone https://github.com/HPI-Information-Systems/DQ4AI.git {DQ4AI_DIR}
    print('DQ4AI 클론 완료')
else:
    print('DQ4AI 이미 존재 — 스킵')

# DQ4AI의 data/clean 디렉토리 생성
os.makedirs(f'{DQ4AI_DIR}/data/clean', exist_ok=True)

DQ4AI 이미 존재 — 스킵


## 1. 데이터 다운로드

In [3]:
# ============================================================
# 1-1. SouthGermanCredit (GitHub mirror — UCI 서버 불안정 대비)
# ============================================================
import pandas as pd

sgc_url = 'https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv'
df_sgc = pd.read_csv(sgc_url)
sgc_path = f'{RAW_DIR}/SouthGermanCredit.csv'
df_sgc.to_csv(sgc_path, index=False)
print(f'SouthGermanCredit: {df_sgc.shape} → {sgc_path}')
df_sgc.head()

SouthGermanCredit: (1000, 21) → /content/drive/MyDrive/capstone/dsc/data/raw/SouthGermanCredit.csv


,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk
0,... < 100 DM,6,critical account/other credits existing,domestic appliances,1169,unknown/no savings account,... >= 7 years,4,male : single,none,...,real estate,67,none,own,2,skilled employee/official,1,yes,yes,1
1,0 <= ... < 200 DM,48,existing credits paid back duly till now,domestic appliances,5951,... < 100 DM,1 <= ... < 4 years,2,female : divorced/separated/married,none,...,real estate,22,none,own,1,skilled employee/official,1,no,yes,0
2,no checking account,12,critical account/other credits existing,retraining,2096,... < 100 DM,4 <= ... < 7 years,2,male : single,none,...,real estate,49,none,own,1,unskilled - resident,2,no,yes,1
3,... < 100 DM,42,existing credits paid back duly till now,radio/television,7882,... < 100 DM,4 <= ... < 7 years,2,male : single,guarantor,...,building society savings agreement/life insurance,45,none,for free,1,skilled employee/official,2,no,yes,1
4,... < 100 DM,24,delay in paying off in the past,car (new),4870,... < 100 DM,1 <= ... < 4 years,3,male : single,none,...,unknown/no property,53,none,for free,2,skilled employee/official,2,no,yes,0


In [4]:
# ============================================================
# 1-2. letter (OpenML — UCI 서버 불안정 대비)
# ============================================================
letter_url = 'https://www.openml.org/data/get_csv/6/letter.csv'
df_letter = pd.read_csv(letter_url)
# OpenML에서는 타겟 컬럼명이 'class' → 'lettr'로 변환
df_letter = df_letter.rename(columns={'class': 'lettr'})
letter_path = f'{RAW_DIR}/letter.csv'
df_letter.to_csv(letter_path, index=False)
print(f'letter: {df_letter.shape} → {letter_path}')
df_letter.head()

letter: (20000, 17) → /content/drive/MyDrive/capstone/dsc/data/raw/letter.csv


,x-box,y-box,width,high,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx,lettr
0,2,4,4,3,2,7,8,2,9,11,7,7,1,8,5,6,Z
1,4,7,5,5,5,5,9,6,4,8,7,9,2,9,7,10,P
2,7,10,8,7,4,8,8,5,10,11,2,8,2,5,5,10,S
3,4,9,5,7,4,7,7,13,1,7,6,8,3,8,0,8,H
4,6,7,8,5,4,7,6,3,7,10,7,9,3,8,3,7,H


In [5]:
# ============================================================
# 1-3. TelcoCustomerChurn (IBM GitHub — Kaggle API 불필요)
# ============================================================
telco_url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df_telco = pd.read_csv(telco_url)
telco_path = f'{RAW_DIR}/TelcoCustomerChurn.csv'
df_telco.to_csv(telco_path, index=False)
print(f'TelcoCustomerChurn: {df_telco.shape} → {telco_path}')
df_telco.head()

TelcoCustomerChurn: (7043, 21) → /content/drive/MyDrive/capstone/dsc/data/raw/TelcoCustomerChurn.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [6]:
# ============================================================
# 1-4. DQ4AI data/clean/에 원본 복사 (polluter 호환용)
# ============================================================
import shutil

for name in ['SouthGermanCredit', 'TelcoCustomerChurn', 'letter']:
    src = f'{RAW_DIR}/{name}.csv'
    dst = f'{DQ4AI_DIR}/data/clean/{name}.csv'
    shutil.copy(src, dst)
    print(f'  {name}.csv → DQ4AI/data/clean/')

print('\n원본 데이터 준비 완료')

  SouthGermanCredit.csv → DQ4AI/data/clean/
  TelcoCustomerChurn.csv → DQ4AI/data/clean/
  letter.csv → DQ4AI/data/clean/

원본 데이터 준비 완료


In [7]:
# ============================================================
# 1-5. 데이터 요약 확인
# ============================================================
DATASETS = {
    'TelcoCustomerChurn': {
        'path': f'{RAW_DIR}/TelcoCustomerChurn.csv',
        'target': 'Churn',
        'numerical_cols': ['tenure', 'MonthlyCharges', 'TotalCharges'],
        'categorical_cols': [
            'gender', 'SeniorCitizen', 'Partner', 'Dependents',
            'PhoneService', 'MultipleLines', 'InternetService',
            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies',
            'Contract', 'PaperlessBilling', 'PaymentMethod'
        ],
        'placeholder_numerical': -1,
        'placeholder_categorical': 'empty',
    },
    'SouthGermanCredit': {
        'path': f'{RAW_DIR}/SouthGermanCredit.csv',
        'target': 'credit_risk',
        'numerical_cols': ['duration', 'amount', 'age'],
        'categorical_cols': [
            'status', 'credit_history', 'purpose', 'savings',
            'employment_duration', 'installment_rate', 'personal_status_sex',
            'other_debtors', 'present_residence', 'property',
            'other_installment_plans', 'housing', 'number_credits',
            'job', 'people_liable', 'telephone', 'foreign_worker'
        ],
        'placeholder_numerical': -1,
        'placeholder_categorical': 'missing',
    },
    'letter': {
        'path': f'{RAW_DIR}/letter.csv',
        'target': 'lettr',
        'numerical_cols': [
            'x-box', 'y-box', 'width', 'high', 'onpix', 'x-bar',
            'y-bar', 'x2bar', 'y2bar', 'xybar', 'x2ybr', 'xy2br',
            'x-ege', 'xegvy', 'y-ege', 'yegvx'
        ],
        'categorical_cols': [],
        'placeholder_numerical': -1,
        'placeholder_categorical': None,
    },
}

for name, meta in DATASETS.items():
    df = pd.read_csv(meta['path'])
    print(f'\n=== {name} ===')
    print(f'  Shape: {df.shape}')
    print(f'  Target: {meta["target"]} → {df[meta["target"]].nunique()} classes')
    print(f'  Null 합계: {df.isnull().sum().sum()}')
    print(f'  수치형: {len(meta["numerical_cols"])}개, 범주형: {len(meta["categorical_cols"])}개')


=== TelcoCustomerChurn ===
  Shape: (7043, 21)
  Target: Churn → 2 classes
  Null 합계: 0
  수치형: 3개, 범주형: 16개

=== SouthGermanCredit ===
  Shape: (1000, 21)
  Target: credit_risk → 2 classes
  Null 합계: 0
  수치형: 3개, 범주형: 17개

=== letter ===
  Shape: (20000, 17)
  Target: lettr → 26 classes
  Null 합계: 0
  수치형: 16개, 범주형: 0개


## 2. DSC 스코어링 엔진

7개 지표를 계산하여 0~100 점수 + A/B/C/D 등급을 산출한다.

In [8]:
# ============================================================
# 2-1. DSC 스코어링 엔진 정의 (v3 — outlier reference 고정 + value_accuracy 신설)
# ============================================================
import numpy as np
import re
from scipy import stats as sp_stats


def calc_completeness(df, target_col, placeholder_numerical=-1, placeholder_categorical='empty'):
    """결측치 + placeholder 비율. 1=완전, 0=전부 결측."""
    feature_df = df.drop(columns=[target_col], errors='ignore')
    total_cells = feature_df.shape[0] * feature_df.shape[1]
    if total_cells == 0:
        return 1.0
    missing_count = feature_df.isnull().sum().sum()
    if placeholder_numerical is not None:
        for col in feature_df.select_dtypes(include=[np.number]).columns:
            missing_count += (feature_df[col] == placeholder_numerical).sum()
    if placeholder_categorical is not None:
        for col in feature_df.select_dtypes(include=['object', 'category']).columns:
            ph = (placeholder_categorical.get(col, 'empty')
                  if isinstance(placeholder_categorical, dict)
                  else placeholder_categorical)
            missing_count += (feature_df[col].astype(str) == str(ph)).sum()
    return 1.0 - (missing_count / total_cells)


def calc_uniqueness(df, target_col):
    """중복 행 비율. 1=전부 유일."""
    n = len(df)
    if n <= 1:
        return 1.0
    return 1.0 - (df.duplicated().sum() / n)


def calc_validity(df, target_col, numerical_cols, categorical_cols):
    """타입/형식 유효성."""
    scores = []
    for col in numerical_cols:
        if col not in df.columns:
            continue
        converted = pd.to_numeric(df[col], errors='coerce')
        total = len(df[col].dropna())
        scores.append(converted.notna().sum() / total if total > 0 else 1.0)
    for col in categorical_cols:
        if col not in df.columns:
            continue
        s = df[col].dropna().astype(str)
        if len(s) == 0:
            scores.append(1.0)
            continue
        valid = s.apply(lambda x: 0 < len(x.strip()) < 200)
        scores.append(valid.mean())
    return float(np.mean(scores)) if scores else 1.0


def calc_consistency(df, target_col, categorical_cols):
    """범주형 표현 일관성 — 오염 시그니처 행 비율 측정."""
    if not categorical_cols:
        return 1.0
    scores = []
    for col in categorical_cols:
        if col not in df.columns:
            continue
        s = df[col].dropna().astype(str)
        if len(s) == 0:
            scores.append(1.0)
            continue
        has_suffix = s.apply(lambda x: bool(re.search(r'-\d+$', x)))
        scores.append(1.0 - has_suffix.mean())
    return float(np.mean(scores)) if scores else 1.0


def calc_outlier_ratio(df, target_col, numerical_cols, reference_df=None):
    """IQR 기반 outlier가 아닌 비율.
    v3: reference_df의 IQR을 고정 기준으로 사용 → 노이즈가 IQR을 부풀려 outlier가
    줄어드는 자기참조 함정 회피."""
    if not numerical_cols:
        return 1.0
    scores = []
    for col in numerical_cols:
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(s) < 4:
            scores.append(1.0)
            continue
        if reference_df is not None and col in reference_df.columns:
            ref = pd.to_numeric(reference_df[col], errors='coerce').dropna()
            if len(ref) >= 4:
                q1, q3 = ref.quantile(0.25), ref.quantile(0.75)
            else:
                q1, q3 = s.quantile(0.25), s.quantile(0.75)
        else:
            q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            scores.append(1.0)
            continue
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_count = ((s < lower) | (s > upper)).sum()
        scores.append(1.0 - outlier_count / len(s))
    return float(np.mean(scores)) if scores else 1.0


def calc_class_balance(df, target_col):
    """클래스 균형 — 최소 비율 / 이상 비율."""
    counts = df[target_col].value_counts()
    n_classes = len(counts)
    if n_classes <= 1:
        return 1.0
    min_ratio = counts.min() / counts.sum()
    ideal_ratio = 1.0 / n_classes
    return min(min_ratio / ideal_ratio, 1.0)


def calc_feature_correlation(df, target_col, numerical_cols, threshold=0.95):
    """고상관(>threshold) 피처 쌍이 없는 비율."""
    cols = [c for c in numerical_cols if c in df.columns]
    if len(cols) < 2:
        return 1.0
    num_df = df[cols].apply(pd.to_numeric, errors='coerce')
    corr = num_df.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    total_pairs = upper.size - upper.isna().sum().sum()
    if total_pairs == 0:
        return 1.0
    high_corr_pairs = (upper > threshold).sum().sum()
    return 1.0 - (high_corr_pairs / total_pairs)


def calc_value_accuracy(df, target_col, numerical_cols, categorical_cols, reference_df=None):
    """값 정확성 — reference 분포 대비 drift.
    수치형: 1 - KS statistic. 범주형: 1 - Total Variation Distance.
    reference 없으면 1.0 (지표 비활성)."""
    if reference_df is None:
        return 1.0
    scores = []
    for col in numerical_cols:
        if col not in df.columns or col not in reference_df.columns:
            continue
        ref = pd.to_numeric(reference_df[col], errors='coerce').dropna()
        cur = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(ref) < 5 or len(cur) < 5:
            scores.append(1.0)
            continue
        ks_stat, _ = sp_stats.ks_2samp(ref, cur)
        scores.append(1.0 - float(ks_stat))
    for col in categorical_cols:
        if col not in df.columns or col not in reference_df.columns:
            continue
        ref_counts = reference_df[col].astype(str).value_counts(normalize=True)
        cur_counts = df[col].astype(str).value_counts(normalize=True)
        all_cats = ref_counts.index.union(cur_counts.index)
        ref_p = ref_counts.reindex(all_cats, fill_value=0)
        cur_p = cur_counts.reindex(all_cats, fill_value=0)
        tvd = 0.5 * (ref_p - cur_p).abs().sum()
        scores.append(1.0 - float(tvd))
    return float(np.mean(scores)) if scores else 1.0


DEFAULT_WEIGHTS = {
    'completeness': 0.20, 'uniqueness': 0.15, 'validity': 0.10,
    'consistency': 0.10, 'outlier_ratio': 0.05,
    'class_balance': 0.05, 'feature_correlation': 0.05,
    'value_accuracy': 0.30,
}


def compute_dsc(df, target_col, numerical_cols, categorical_cols,
                weights=None,
                placeholder_numerical=-1,
                placeholder_categorical='empty',
                reference_df=None):
    """DSC 점수(0~100) + 등급 + 지표별 점수.
    v3: value_accuracy 신설, outlier_ratio reference 고정, 가중치 재배분.
    reference_df: 베이스라인 데이터(보통 clean df). 베이스라인 자체 측정 시 자기 자신을 넘김."""
    w = weights or DEFAULT_WEIGHTS
    metrics = {
        'completeness':        calc_completeness(df, target_col, placeholder_numerical, placeholder_categorical),
        'uniqueness':          calc_uniqueness(df, target_col),
        'validity':            calc_validity(df, target_col, numerical_cols, categorical_cols),
        'consistency':         calc_consistency(df, target_col, categorical_cols),
        'outlier_ratio':       calc_outlier_ratio(df, target_col, numerical_cols, reference_df=reference_df),
        'class_balance':       calc_class_balance(df, target_col),
        'feature_correlation': calc_feature_correlation(df, target_col, numerical_cols),
        'value_accuracy':      calc_value_accuracy(df, target_col, numerical_cols, categorical_cols, reference_df=reference_df),
    }
    score = sum(metrics[k] * w[k] for k in w) * 100
    if score >= 90:   grade = 'A'
    elif score >= 75: grade = 'B'
    elif score >= 60: grade = 'C'
    else:             grade = 'D'
    return {'score': round(score, 2), 'grade': grade,
            **{k: round(v, 4) for k, v in metrics.items()}}


print('DSC 스코어링 엔진 v3 정의 완료')
print('  변경: value_accuracy 신설, outlier_ratio reference 고정, 가중치 재배분')

DSC 스코어링 엔진 v2 정의 완료
  변경: completeness(placeholder감지), consistency(행비율), class_balance(최소비율)


## 3. 베이스라인 DSC 점수

In [9]:
# ============================================================
# 3-1. 원본 데이터 DSC 점수 계산
# ============================================================
baseline_dsc_rows = []

for ds_name, meta in DATASETS.items():
    df = pd.read_csv(meta['path'])
    result = compute_dsc(
        df,
        target_col=meta['target'],
        numerical_cols=meta['numerical_cols'],
        categorical_cols=meta['categorical_cols'],
        placeholder_numerical=meta.get('placeholder_numerical', -1),
        placeholder_categorical=meta.get('placeholder_categorical', 'empty'),
    reference_df=df,
    )
    row = {
        'dataset': ds_name,
        'polluter': 'none',
        'level': 0.0,
        **result,
    }
    baseline_dsc_rows.append(row)
    print(f"\n{ds_name}: DSC={result['score']} ({result['grade']})")
    for k, v in result.items():
        if k not in ('score', 'grade'):
            print(f'  {k}: {v}')

df_baseline_dsc = pd.DataFrame(baseline_dsc_rows)
df_baseline_dsc


TelcoCustomerChurn: DSC=97.65 (A)
  completeness: 1.0
  uniqueness: 1.0
  validity: 0.9999
  consistency: 1.0
  outlier_ratio: 1.0
  class_balance: 0.5307
  feature_correlation: 1.0

SouthGermanCredit: DSC=97.45 (A)
  completeness: 1.0
  uniqueness: 1.0
  validity: 1.0
  consistency: 1.0
  outlier_ratio: 0.945
  class_balance: 0.6
  feature_correlation: 1.0

letter: DSC=98.11 (A)
  completeness: 1.0
  uniqueness: 0.9334
  validity: 1.0
  consistency: 1.0
  outlier_ratio: 0.9671
  class_balance: 0.9542
  feature_correlation: 1.0


,dataset,polluter,level,score,grade,completeness,uniqueness,validity,consistency,outlier_ratio,class_balance,feature_correlation
0,TelcoCustomerChurn,none,0.0,97.65,A,1.0,1.0000,0.9999,1.0,1.0000,0.5307,1.0
1,SouthGermanCredit,none,0.0,97.45,A,1.0,1.0000,1.0000,1.0,0.9450,0.6000,1.0
2,letter,none,0.0,98.11,A,1.0,0.9334,1.0000,1.0,0.9671,0.9542,1.0


## 4. 베이스라인 모델 학습 & 평가

3 데이터셋 × 5 모델 = 15회

In [10]:
# ============================================================
# 4-1. 모델 및 전처리 정의
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')


def get_models():
    return {
        'LogisticRegression': LogisticRegression(
            solver='lbfgs', max_iter=2000, random_state=42, n_jobs=-1
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=100, random_state=42, n_jobs=-1
        ),
        'XGBoost': XGBClassifier(
            n_estimators=100, random_state=42, n_jobs=-1,
            use_label_encoder=False, eval_metric='mlogloss'
        ),
        'SVC': SVC(
            kernel='linear', random_state=42, probability=True
        ),
        'MLP': MLPClassifier(
            hidden_layer_sizes=(100, 100, 100, 100, 100),
            random_state=42, max_iter=1000
        ),
    }


def prepare_data(df, meta):
    """전처리 파이프라인: train/test 분할 + 인코딩 + 스케일링."""
    target = meta['target']
    num_cols = [c for c in meta['numerical_cols'] if c in df.columns]
    cat_cols = [c for c in meta['categorical_cols'] if c in df.columns]
    feature_cols = num_cols + cat_cols

    X = df[feature_cols].copy()
    y = df[target].copy()

    # 수치형 컬럼 강제 변환 (TelcoCustomerChurn의 TotalCharges 등)
    for col in num_cols:
        X[col] = pd.to_numeric(X[col], errors='coerce')

    # 결측치 처리 (베이스라인용 간단 처리)
    for col in num_cols:
        X[col] = X[col].fillna(X[col].median())
    for col in cat_cols:
        X[col] = X[col].fillna('MISSING')

    # 타겟 인코딩
    le = LabelEncoder()
    y = le.fit_transform(y)

    # train/test 분할
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=1, stratify=y
    )

    # 전처리 파이프라인
    transformers = []
    if num_cols:
        transformers.append(('num', StandardScaler(), num_cols))
    if cat_cols:
        transformers.append(('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols))

    preprocessor = ColumnTransformer(transformers, remainder='drop')
    X_train_t = preprocessor.fit_transform(X_train)
    X_test_t = preprocessor.transform(X_test)

    n_classes = len(np.unique(y))

    return X_train_t, X_test_t, y_train, y_test, n_classes, le


def evaluate_model(model, X_test, y_test, n_classes):
    """Accuracy, F1(macro), AUC-ROC 반환."""
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    # AUC-ROC
    try:
        if hasattr(model, 'predict_proba'):
            y_prob = model.predict_proba(X_test)
        else:
            y_prob = model.decision_function(X_test)
        if n_classes == 2:
            if y_prob.ndim == 2:
                y_prob = y_prob[:, 1]
            auc = roc_auc_score(y_test, y_prob)
        else:
            auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
    except Exception:
        auc = np.nan

    return {'accuracy': round(acc, 4), 'f1_macro': round(f1, 4), 'auc_roc': round(auc, 4)}


print('모델 및 전처리 함수 정의 완료')

모델 및 전처리 함수 정의 완료


In [11]:
# ============================================================
# 4-2. 베이스라인 모델 학습 실행 (3 × 5 = 15회)
# ============================================================
from time import time

baseline_perf_rows = []

for ds_name, meta in DATASETS.items():
    print(f'\n=== {ds_name} ===')
    df = pd.read_csv(meta['path'])
    X_train, X_test, y_train, y_test, n_classes, le = prepare_data(df, meta)

    for model_name, model in get_models().items():
        t0 = time()
        model.fit(X_train, y_train)
        scores = evaluate_model(model, X_test, y_test, n_classes)
        elapsed = time() - t0

        row = {
            'dataset': ds_name,
            'polluter': 'none',
            'level': 0.0,
            'model': model_name,
            **scores,
        }
        baseline_perf_rows.append(row)
        print(f'  {model_name:22s} → F1={scores["f1_macro"]:.4f}  Acc={scores["accuracy"]:.4f}  AUC={scores["auc_roc"]:.4f}  ({elapsed:.1f}s)')

df_baseline_perf = pd.DataFrame(baseline_perf_rows)
print('\n베이스라인 학습 완료')


=== TelcoCustomerChurn ===
  LogisticRegression     → F1=0.7389  Acc=0.8084  AUC=0.8349  (2.2s)
  RandomForest           → F1=0.6980  Acc=0.7835  AUC=0.8055  (0.8s)
  XGBoost                → F1=0.7041  Acc=0.7835  AUC=0.8124  (0.2s)
  SVC                    → F1=0.7303  Acc=0.8041  AUC=0.8241  (20.8s)
  MLP                    → F1=0.6792  Acc=0.7509  AUC=0.7632  (33.6s)

=== SouthGermanCredit ===
  LogisticRegression     → F1=0.6991  Acc=0.7600  AUC=0.7926  (1.3s)
  RandomForest           → F1=0.6988  Acc=0.7850  AUC=0.8049  (0.5s)
  XGBoost                → F1=0.7415  Acc=0.7950  AUC=0.8119  (0.3s)
  SVC                    → F1=0.6740  Acc=0.7400  AUC=0.7810  (0.3s)
  MLP                    → F1=0.7235  Acc=0.7700  AUC=0.7590  (4.8s)

=== letter ===
  LogisticRegression     → F1=0.7644  Acc=0.7660  AUC=0.9786  (1.9s)
  RandomForest           → F1=0.9577  Acc=0.9577  AUC=0.9993  (2.2s)
  XGBoost                → F1=0.9592  Acc=0.9593  AUC=0.9996  (4.8s)
  SVC                    → F1=

## 5. 결과 저장

In [12]:
# ============================================================
# 5-1. 베이스라인 결과를 CSV로 저장
# ============================================================
dsc_scores_path = f'{RESULTS_DIR}/dsc_scores.csv'
model_perf_path = f'{RESULTS_DIR}/model_performance.csv'

df_baseline_dsc.to_csv(dsc_scores_path, index=False)
df_baseline_perf.to_csv(model_perf_path, index=False)

print(f'DSC 점수 저장: {dsc_scores_path}')
print(f'모델 성능 저장: {model_perf_path}')
print(f'\n--- 노트북 01 완료 ---')
print(f'다음: 02_pollution_and_dsc.ipynb 실행')

DSC 점수 저장: /content/drive/MyDrive/capstone/dsc/results/dsc_scores.csv
모델 성능 저장: /content/drive/MyDrive/capstone/dsc/results/model_performance.csv

--- 노트북 01 완료 ---
다음: 02_pollution_and_dsc.ipynb 실행


In [13]:
# ============================================================
# 5. 실행 로그 저장
# ============================================================
from datetime import datetime

log_lines = []
log_lines.append('# 노트북 01 실행 로그: 환경 세팅 & 베이스라인')
log_lines.append(f'')
log_lines.append(f'- **실행 시각**: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
log_lines.append(f'- **BASE 경로**: {BASE}')
log_lines.append(f'')

# 데이터셋 요약
log_lines.append('## 1. 데이터 다운로드')
log_lines.append('')
log_lines.append('| 데이터셋 | 행 | 열 | 타겟 | 클래스 수 | Null 합계 |')
log_lines.append('|---|---|---|---|---|---|')
for ds_name, meta in DATASETS.items():
    df_tmp = pd.read_csv(meta['path'])
    n_classes = df_tmp[meta['target']].nunique()
    n_null = df_tmp.isnull().sum().sum()
    log_lines.append(f'| {ds_name} | {df_tmp.shape[0]:,} | {df_tmp.shape[1]} | {meta["target"]} | {n_classes} | {n_null} |')
log_lines.append('')

# DSC 베이스라인
log_lines.append('## 2. 베이스라인 DSC 점수')
log_lines.append('')
log_lines.append('| 데이터셋 | DSC Score | 등급 | completeness | uniqueness | validity | consistency | outlier_ratio | class_balance | feature_correlation |')
log_lines.append('|---|---|---|---|---|---|---|---|---|---|')
for row in baseline_dsc_rows:
    log_lines.append(f'| {row["dataset"]} | {row["score"]} | {row["grade"]} | {row["completeness"]} | {row["uniqueness"]} | {row["validity"]} | {row["consistency"]} | {row["outlier_ratio"]} | {row["class_balance"]} | {row["feature_correlation"]} |')
log_lines.append('')

# 모델 성능
log_lines.append('## 3. 베이스라인 모델 성능')
log_lines.append('')
log_lines.append('| 데이터셋 | 모델 | Accuracy | F1(macro) | AUC-ROC |')
log_lines.append('|---|---|---|---|---|')
for row in baseline_perf_rows:
    log_lines.append(f'| {row["dataset"]} | {row["model"]} | {row["accuracy"]} | {row["f1_macro"]} | {row["auc_roc"]} |')
log_lines.append('')

log_lines.append('## 4. 산출물')
log_lines.append('')
log_lines.append(f'-  — 베이스라인 DSC 점수 {len(baseline_dsc_rows)}건')
log_lines.append(f'-  — 베이스라인 모델 성능 {len(baseline_perf_rows)}건')
log_lines.append('')
log_lines.append('---')
log_lines.append('*이 로그는 노트북 01 실행 시 자동 생성됨*')

log_path = f'{RESULTS_DIR}/01_execution_log.md'
with open(log_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(log_lines))
print(f'실행 로그 저장: {log_path}')

실행 로그 저장: /content/drive/MyDrive/capstone/dsc/results/01_execution_log.md
